# 03 — Connect the Bigdata.com MCP Server

This is the **external data** side of the demo. Bigdata.com exposes a remote
**Model Context Protocol (MCP)** server at `https://mcp.bigdata.com/` that streams
real-time financial intelligence — news, filings, earnings transcripts, company
tearsheets, events. Because it speaks MCP, the agent discovers its tools
automatically; you never hand-code an API wrapper per capability.

There are two ways to wire it into Databricks. This notebook covers both:

| Path | What it is | When to use |
|---|---|---|
| **A. Governed — MCP Service** | Register Bigdata.com behind a Unity Catalog HTTP connection and expose it through **Unity AI Gateway** as an MCP Service. Credentials, per-user `EXECUTE` grants, and full audit logging live in Unity Catalog. | Production / shared workspaces. The recommended, governed path. |
| **B. Direct** | The agent connects straight to `https://mcp.bigdata.com/` with the API key from a Databricks secret. | Fastest to run; great for a self-contained demo or a single developer. |

The rest of the demo works with **either** path — notebook `05` reads a single
config flag to decide which one the agent uses.

In [0]:
%pip install -U "mcp>=1.9" langchain-mcp-adapters
dbutils.library.restartPython()

In [0]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"
SECRET_SCOPE = "bigdata"
SECRET_KEY = "api_key"

BIGDATA_MCP_URL = "https://mcp.bigdata.com/"
MCP_SERVICE_NAME = f"{CATALOG}.{SCHEMA}.bigdata_mcp"   # used by Path A
CONNECTION_NAME = f"{CATALOG}.{SCHEMA}.bigdata_http"    # used by Path A

## Path B (Direct) — connectivity test

Run this first regardless of which path you deploy with: it confirms your API key
works and shows the tools Bigdata.com exposes over MCP. The agent in notebook `05`
uses this exact `MultiServerMCPClient` pattern when `USE_MCP_SERVICE = False`.

In [0]:
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# Databricks notebooks already run inside an event loop, so we use top-level `await`
# in the cells below. nest_asyncio makes that safe and avoids the error
# "asyncio.run() cannot be called from a running event loop".
nest_asyncio.apply()

BIGDATA_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# Build the Bigdata.com MCP client ONCE and reuse it in every cell below.
# (Remote MCP over streamable HTTP, authenticated with the x-api-key header.)
bigdata_client = MultiServerMCPClient(
    {
        "bigdata": {
            "url": BIGDATA_MCP_URL,
            "transport": "streamable_http",
            "headers": {"x-api-key": BIGDATA_API_KEY},
        }
    }
)

In [0]:
# Discover the Bigdata.com tools ONCE; reuse `bigdata_tools` / `tools_by_name` below.
bigdata_tools = await bigdata_client.get_tools()
tools_by_name = {t.name: t for t in bigdata_tools}

print(f"Bigdata.com MCP exposed {len(bigdata_tools)} tools:\n")
for t in bigdata_tools:
    print(f"  - {t.name}: {(t.description or '')[:90]}...")

You should see a broad, evolving tool set — for example:

| Tool | What it does |
|---|---|
| `bigdata_search` | Search news, filings, transcripts, podcasts, and research |
| `find_securities` | Resolve companies, ETFs, and funds by name/ticker/ISIN (**replaces** `find_companies`) |
| `get_securities` | Batch-resolve exact identifiers (ISIN/CUSIP/SEDOL), 1–50 at a time |
| `bigdata_company_tearsheet` | Company financials + analyst coverage |
| `bigdata_etf_tearsheet` | ETF holdings, performance, allocations |
| `bigdata_country_tearsheet` | Country macro snapshot (calendar, indices, FX) |
| `bigdata_market_tearsheet` | Cross-asset market snapshot (energy, equities, FX, ETFs) |
| `bigdata_sentiment_tearsheet` | Real-time media sentiment + narratives |
| `bigdata_events_calendar` | Earnings / conference-call calendar |
| `bigdata_screen_credit_factor` / `bigdata_get_credit_factor` | Credit-risk factor screening |
| `bigdata_screen_fund_managers` | Institutional ownership / fund-manager screening |

New Bigdata.com tools appear here automatically — no code change required, which is
the whole point of connecting over MCP.

## Path B — call a tool end to end

In [0]:
# Reuse the client + tools created above — no new client, no asyncio.run().
search = tools_by_name["bigdata_search"]
result = await search.ainvoke({"search_text": "NVIDIA data center revenue growth 2025"})
print(str(result)[:1500])

---
## Path A (Governed) — register Bigdata.com as an MCP Service

This is the flagship, governed integration. It mirrors what the Snowflake demo
does with an External Access Integration + Secret — but with Unity Catalog and AI
Gateway doing credential management, permissioning, and audit. **MCP Services are
managed via the UI or REST API (no SQL DDL yet).**

### A.1 — Create the Unity Catalog HTTP connection

In **Catalog → Connections → Create connection**:

| Field | Value |
|---|---|
| Connection name | `bigdata_http` |
| Connection type | **HTTP** |
| Host | `https://mcp.bigdata.com` |
| Port | `443` |
| Base path | `/` |
| Auth type | **Bearer token** *(see note)* |

> **Header note:** Bigdata.com authenticates with the `x-api-key` header. If your
> workspace's HTTP connection UI offers a **custom header** / API-key option, set
> header `x-api-key` = your key. If it only offers Bearer, add the custom header on
> the MCP Service config instead (`config.headers`), as shown in the REST call below.

The equivalent REST call (fill in `<host>` and `<key>`):

```bash
databricks connections create --json '{
  "name": "bigdata_http",
  "connection_type": "HTTP",
  "options": {
    "host": "https://mcp.bigdata.com",
    "port": "443",
    "base_path": "/",
    "bearer_token": "<key>"
  }
}'
```

### A.2 — Register the MCP Service (REST API)

```bash
databricks api post \
  "/api/2.1/unity-catalog/mcp-services?parent=schemas/bigdata_demo.financial_intelligence&mcp_service_id=bigdata_mcp" \
  --json '{
    "comment": "Bigdata.com financial intelligence MCP",
    "config": {
      "connection": { "name": "connections/bigdata_demo.financial_intelligence.bigdata_http" },
      "headers": { "x-api-key": "<key>" },
      "include_tool_selectors": []
    }
  }'
```

### A.3 — Grant EXECUTE to the agent's principal / users

```sql
GRANT EXECUTE ON MCP SERVICE bigdata_demo.financial_intelligence.bigdata_mcp TO `account users`;
```

Invoking an MCP Service needs **only** `EXECUTE` on the service — callers never get
direct access to the underlying connection or the API key.

### Resulting server URL (used by notebook `05` when `USE_MCP_SERVICE = True`)
```
https://<workspace-hostname>/ai-gateway/mcp-services/bigdata_demo.financial_intelligence.bigdata_mcp
```

### A.4 — Verify the MCP Service (Databricks-authenticated client)

Once registered, the governed service is reachable with `DatabricksMCPClient`, which
handles Databricks OAuth for you. Uncomment and run after completing A.1–A.3.

In [0]:
# from databricks_mcp import DatabricksMCPClient
# from databricks.sdk import WorkspaceClient
#
# ws = WorkspaceClient()
# service_url = f"{ws.config.host}/ai-gateway/mcp-services/{MCP_SERVICE_NAME}"
# client = DatabricksMCPClient(server_url=service_url, workspace_client=ws)
# print([t.name for t in client.list_tools()])

## Done

External financial intelligence is connected. Optionally set up an AI/BI **Genie**
space (**`04_genie_space`**) for open-ended structured querying, then build the
agent in **`05_build_deploy_agent`**.